# Глава 8. ML для бизнес-решений — практика в Jupyter

Те же задачи, что на сайте курса, но в своём Jupyter: так вы привыкаете к блокноту, горячим клавишам и своему окружению. Проверка — та же, что в браузере.

**Как запустить** (один раз поставьте [uv](https://docs.astral.sh/uv/), см. урок «Своё рабочее место — uv и Jupyter» главы 2). В терминале, в папке курса:

```sh
uv run --with-requirements site/notebooks/requirements.txt jupyter lab site/notebooks
```

Без uv: `python3 -m venv .venv`, затем `.venv/bin/pip install -r site/notebooks/requirements.txt` (в Windows — `.venv\Scripts\pip`) и `.venv/bin/jupyter lab site/notebooks`. Откройте этот файл в папке `site/notebooks` — рядом с ним лежит расширение `dsc.py`, без него проверка не заработает.

**Как решать**

1. Запустите ячейку ниже с `%load_ext dsc` — она подключает проверку. После перезапуска ядра запустите её снова.
2. В ячейке задачи первая строка — `%%task глава/урок/id`. Её не трогайте, пишите код ниже и запускайте **Shift+Enter**. Под выводом появится «✓ Верно» или подсказка, что не так.
3. Переменные живут между ячейками, как в любом блокноте. Исключение — переменные, в которые задача просит положить ответ: перед запуском ячейки они стираются, чтобы проверка видела результат именно этого запуска. Подготовка задачи (если она есть) выполняется перед вашим кодом при каждом запуске.
4. Застряли — `dsc.hint('id')` покажет следующую подсказку, `dsc.solution('id', sure=True)` — эталон с разбором. `dsc.progress()` — что решено в этом сеансе.
5. Файлы данных берутся по имени, как на сайте: `pd.read_csv('orders.csv')`. Код работает в папке `site/notebooks/_work` — там копия `site/data`, и туда же попадает всё, что сохраняет ваш код.

Прогресс сайта и блокнота не связаны: отметки о решённых задачах на сайте ставятся только там. Чтобы начать с чистого листа, скачайте блокнот заново или возьмите его из архива курса.

In [ ]:
%load_ext dsc

## Урок 1. Из прогноза — в деньги

Модель оттока из главы 7 говорит, кто, вероятно, уйдёт. Бизнесу нужен другой ответ: кому дать бонус, чтобы заработать. Переводим матрицу ошибок в рубли, выбираем порог по прибыли, проверяем, насколько решение зависит от допущений. Как проверить это обещание пилотом и разложить выручку на дерево метрик — в двух следующих уроках.

Теория — на сайте курса: `ch08/01-value.html`. Задачи «собери код из строк» (1) остались на сайте.

### Прибыль по матрице ошибок · разминка

Кампания отправила бонусы 150 гостям, из них 90 действительно собирались уйти. Выгода от TP — 400 ₽, бонус стоит 200 ₽. Сохраните в `fp` число лишних бонусов, а в `profit` — ожидаемую прибыль кампании.

*Подсказки (2): `dsc.hint('cost-matrix')`*

In [ ]:
%%task ch08/01-value/cost-matrix
tp = 90
sent = 150

### Порог безубыточности · разминка

Новое предложение: бесплатный напиток за 150 ₽ удерживает 20 % уходящих, удержанный гость приносит 2400 ₽. Сохраните в `gain` ожидаемую выгоду от удержания одного уходящего (до вычета цены напитка), а в `threshold` — порог безубыточности по вероятности ухода (3 знака).

*Подсказки (2): `dsc.hint('break-even')`*

In [ ]:
%%task ch08/01-value/break-even
cost = 150
retain = 0.20
value = 2400

### Восемь гостей · разминка

Повторите «Разберём вместе» на своих данных. Для восьми гостей даны прогнозы `p` и факт `y`. Выгода от TP — 400 ₽, бонус — 200 ₽. Сохраните прибыль при пороге 0.5 в `profit_50`, при пороге 1/3 — в `profit_33`, и `extra` — сколько бонусов добавилось при снижении порога.

*Подсказки (2): `dsc.hint('toy-guests')`*

In [ ]:
%%task ch08/01-value/toy-guests
import numpy as np
p = np.array([0.85, 0.70, 0.55, 0.42, 0.38, 0.31, 0.25, 0.12])
y = np.array([1, 0, 1, 1, 0, 0, 1, 0])

### Правило против модели · база

Сравните на тестовой выборке три стратегии: модель с порогом 0.5, модель с порогом 0.3 и правило «не был больше 30 дней» (`days_since_last > 30`). Сохраните прибыли в `p50`, `p30`, `rule30` (функция `profit` уже есть в заготовке) и в `best` — название лучшей: `"0.5"`, `"0.3"` или `"правило"`.

*Подсказки (3): `dsc.hint('rule-vs-model')`*

In [ ]:
%%task ch08/01-value/rule-vs-model
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

g = pd.read_csv("guests_churn.csv")
y = g["churned"]
X = pd.get_dummies(g.drop(columns=["guest_id", "churned", "card_blocked"]), dtype=int).fillna({"avg_check": 0})
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
model = RandomForestClassifier(n_estimators=100, min_samples_leaf=5, random_state=0).fit(X_train, y_train)
p = model.predict_proba(X_test)[:, 1]

def profit(send):
    tp = (send & (y_test == 1)).sum()
    fp = (send & (y_test == 0)).sum()
    return int(400 * tp - 200 * fp)

### Средняя и предельная отдача · база

Финансист говорит: «При пороге 0.5 каждый бонус окупается лучше — зачем снижать?». Проверьте. Для порогов 0.5 и 0.35 сохраните `roi_50` и `roi_35` — прибыль, делённую на расходы на бонусы (200 × число бонусов), 3 знака. Затем возьмите только **дополнительных** гостей — с прогнозом выше 0.35, но не выше 0.5: сохраните их число `n_extra`, сколько из них ушли (`tp_extra`), и их вклад в прибыль `profit_extra`.

*Подсказки (2): `dsc.hint('ch08/01-value/margin')`*

In [ ]:
%%task ch08/01-value/margin
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

g = pd.read_csv("guests_churn.csv")
y = g["churned"]
X = pd.get_dummies(g.drop(columns=["guest_id", "churned", "card_blocked"]), dtype=int).fillna({"avg_check": 0})
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
model = RandomForestClassifier(n_estimators=100, min_samples_leaf=5, random_state=0).fit(X_train, y_train)
p = model.predict_proba(X_test)[:, 1]

def profit(send):
    tp = (send & (y_test == 1)).sum()
    fp = (send & (y_test == 0)).sum()
    return int(400 * tp - 200 * fp)

### Порог по прибыли · база

Переберите пороги от 0.05 до 0.95 с шагом 0.05 (`np.arange(0.05, 0.96, 0.05).round(2)`) и найдите порог с наибольшей прибылью. Сохраните таблицу `curve` (DataFrame со столбцами `threshold` и `profit`), лучший порог в `best_t` и его прибыль в `best_profit`. Если максимум достигается на нескольких порогах, берите наименьший (так делает `idxmax`).

*Подсказки (2): `dsc.hint('ch08/01-value/best-threshold')`*

In [ ]:
%%task ch08/01-value/best-threshold
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

g = pd.read_csv("guests_churn.csv")
y = g["churned"]
X = pd.get_dummies(g.drop(columns=["guest_id", "churned", "card_blocked"]), dtype=int).fillna({"avg_check": 0})
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
model = RandomForestClassifier(n_estimators=100, min_samples_leaf=5, random_state=0).fit(X_train, y_train)
p = model.predict_proba(X_test)[:, 1]

def profit(send):
    tp = (send & (y_test == 1)).sum()
    fp = (send & (y_test == 0)).sum()
    return int(400 * tp - 200 * fp)

### Что если бонус работает хуже · база

Посчитайте лучший порог и прибыль при трёх вариантах удержания: 15 %, 25 % и 35 %. Выгода от TP = удержание × 2400 − 200, расход на FP = 200. Пороги — те же `np.arange(0.05, 0.96, 0.05).round(2)`. Сохраните DataFrame `sens` со столбцами `retain`, `best_t`, `best_profit` (по строке на вариант, в порядке 0.15, 0.25, 0.35).

*Подсказки (2): `dsc.hint('sensitivity')`*

In [ ]:
%%task ch08/01-value/sensitivity
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

g = pd.read_csv("guests_churn.csv")
y = g["churned"]
X = pd.get_dummies(g.drop(columns=["guest_id", "churned", "card_blocked"]), dtype=int).fillna({"avg_check": 0})
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
model = RandomForestClassifier(n_estimators=100, min_samples_leaf=5, random_state=0).fit(X_train, y_train)
p = model.predict_proba(X_test)[:, 1]
ts = np.arange(0.05, 0.96, 0.05).round(2)

### Калибровка у порога · посложнее

Проверьте калибровку леса около порога: возьмите гостей теста с прогнозом от 0.25 до 0.45 включительно. Сохраните в `n_near` их число, в `mean_p` — средний прогноз, в `fact` — долю реально ушедших (оба — 3 знака) и в `gap` — `fact - mean_p` (3 знака).

*Подсказки (2): `dsc.hint('ch08/01-value/calib')`*

In [ ]:
%%task ch08/01-value/calib
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

g = pd.read_csv("guests_churn.csv")
y = g["churned"]
X = pd.get_dummies(g.drop(columns=["guest_id", "churned", "card_blocked"]), dtype=int).fillna({"avg_check": 0})
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
model = RandomForestClassifier(n_estimators=100, min_samples_leaf=5, random_state=0).fit(X_train, y_train)
p = model.predict_proba(X_test)[:, 1]

### Ценность гостя разная · со звёздочкой

Допущение «каждый удержанный приносит 2400 ₽» грубое. Пусть ценность гостя за полгода = 0.3 × `avg_check` × `visits_90d` × 2 (маржа 30 %, два квартала). Отправляйте бонус, если p × 0.25 × ценность > 200. Сохраните в `n_value` число бонусов по такому правилу, в `profit_value` — ожидаемую прибыль (сумма 0.25 × ценность по ушедшим получателям минус 200 × число получателей, округлите до целого), а в `share_low` — долю ушедших гостей теста с ценностью меньше 2400 ₽ (3 знака).

*Подсказки (2): `dsc.hint('ch08/01-value/per-guest')` · задача по желанию*

In [ ]:
%%task ch08/01-value/per-guest
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

g = pd.read_csv("guests_churn.csv")
y = g["churned"]
X = pd.get_dummies(g.drop(columns=["guest_id", "churned", "card_blocked"]), dtype=int).fillna({"avg_check": 0})
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
model = RandomForestClassifier(n_estimators=100, min_samples_leaf=5, random_state=0).fit(X_train, y_train)
p = model.predict_proba(X_test)[:, 1]

## Урок 2. Пилот и P&L: что обещала модель и что вышло

Прибыль из прошлого урока — обещание, посчитанное по допущению «бонус удерживает 25 % уходящих». Сверяем его с пилотом на контрольной группе, видим, сколько бонус приносит на самом деле, и собираем результат в P&L, который примет финансист.

Теория — на сайте курса: `ch08/01a-pilot.html`.

### Эффект бонуса на один бонус · разминка

Разминка перед P&L. По `bonus_pilot.csv` посчитайте `delta` — на сколько бонус снизил долю ушедших (доля в контроле минус доля в группе «бонус», 3 знака) и `online` — прибыль на один бонус по пилоту: `delta × 2400 − 200`, целые рубли (считайте по неокруглённой разности).

*Подсказки (2): `dsc.hint('pilot-delta')`*

In [ ]:
%%task ch08/01a-pilot/pilot-delta
import pandas as pd

b = pd.read_csv("bonus_pilot.csv")

### Кого на самом деле удержал бонус · разминка

Повторите картинку «Пилот в гостях» кодом. Сохраните `stayed_bonus` — сколько гостей группы «бонус» остались (`churned == 0`); `stayed_anyway` — сколько из них остались бы и без бонуса: доля оставшихся в контроле × число гостей в группе «бонус», округлите до целого; `extra` — удержанные сверх контроля (`stayed_bonus − stayed_anyway`).

*Подсказки (2): `dsc.hint('pilot-extra')`*

In [ ]:
%%task ch08/01a-pilot/pilot-extra
import pandas as pd

b = pd.read_csv("bonus_pilot.csv")

### Пилот и P&L · база

Разность долей `delta` вы уже считали в разминке. Теперь сравните её с обещанием и соберите P&L. По `bonus_pilot.csv` посчитайте `offline` — что обещал офлайн-расчёт на один бонус (средний `p_model` в группе «бонус» × 0.25 × 2400 − 200, целые рубли). Затем соберите P&L для 8 000 бонусов: словарь `pnl` с ключами `"маржа"`, `"бонусы"` (расход со знаком минус) и `"итог"`, в рублях, целыми. Для `pnl` используйте неокруглённую разность долей «контроль минус бонус».

*Подсказки (3): `dsc.hint('pilot-pnl')`*

In [ ]:
%%task ch08/01a-pilot/pilot-pnl
import pandas as pd

b = pd.read_csv("bonus_pilot.csv")

### P&L с диапазоном · посложнее

Финансист просит P&L на **3 000** бонусов с 90-процентным интервалом эффекта (z = 1.645 вместо 1.96). По `bonus_pilot.csv` посчитайте итог за полгода в трёх вариантах — `total_est` (по оценке пилота), `total_low` (нижняя граница эффекта) и `total_high` (верхняя), в **тысячах** рублей, округлённых до целых. В `safe` сохраните `True`, если кампания в плюсе даже при нижней границе.

*Подсказки (3): `dsc.hint('pnl-range')`*

In [ ]:
%%task ch08/01a-pilot/pnl-range
import numpy as np
import pandas as pd

b = pd.read_csv("bonus_pilot.csv")
n, z = 3000, 1.645

### Какой эффект окупает бонус · со звёздочкой

Каким должно быть снижение оттока, чтобы бонус хотя бы окупился? Найдите `delta_min` — эффект, при котором прибыль на один бонус равна нулю (`delta × 2400 − 200 = 0`, 4 знака). Затем оцените, насколько уверенно пилот выше этой границы: `z_gap` — на сколько стандартных ошибок эффект пилота больше `delta_min` (2 знака). Стандартная ошибка — как в P&L-памятке.

*Подсказки (3): `dsc.hint('min-effect')` · задача по желанию*

In [ ]:
%%task ch08/01a-pilot/min-effect
import numpy as np
import pandas as pd

b = pd.read_csv("bonus_pilot.csv")

## Урок 3. Дерево метрик: откуда взялось изменение выручки

Финансовый директор видит: гостей больше, а выручка упала. Раскладываем выручку сети на гостей, конверсию и средний чек методом цепных подстановок и находим, какая часть дерева и какие кофейни дали падение. Это инструмент финансового анализа, который пригодится, когда нужно объяснить, откуда взялось изменение выручки.

Теория — на сайте курса: `ch08/01b-metric-tree.html`. Задачи «собери код из строк» (1) остались на сайте.

### Цепочка подстановок руками · разминка

Кофейня в августе: 1200 гостей на входе, конверсия 0.65, средний чек 340 ₽. В сентябре: 1260 гостей, конверсия 0.60, чек 345 ₽. Сохраните выручку августа и сентября в `rev_aug` и `rev_sep`, а вклады множителей (порядок «гости → конверсия → чек») — в `d_guests`, `d_conv` и `d_check`. Округлите всё до целых рублей.

*Подсказки (3): `dsc.hint('tree-calc')`*

In [ ]:
%%task ch08/01b-metric-tree/tree-calc
g0, cr0, ac0 = 1200, 0.65, 340    # август
g1, cr1, ac1 = 1260, 0.60, 345    # сентябрь

### Другой порядок подстановок · база

Возьмите кофейню из «Разберём вместе» (август: 1000 гостей, конверсия 0.70, чек 350 ₽; сентябрь: 1100, 0.63, 335 ₽) и посчитайте вклады в **обратном** порядке — «чек → конверсия → гости». Сохраните `d_check`, `d_conv`, `d_guests` (целые рубли) и `total` — их сумму.

*Подсказки (3): `dsc.hint('order-swap')`*

In [ ]:
%%task ch08/01b-metric-tree/order-swap
g0, cr0, ac0 = 1000, 0.70, 350    # август
g1, cr1, ac1 = 1100, 0.63, 335    # сентябрь

### Дерево сети: август против сентября · база

По `shop_funnel.csv` разложите изменение выручки **сети** с августа на сентябрь. Сохраните Series `parts` с индексом `["гости", "конверсия", "чек"]` и вкладами в рублях (округлите до целых). Затем найдите кофейню с самым большим **падением** конверсии и сохраните её `shop_id` в `worst_shop`, а само падение (сентябрь минус август, 3 знака, отрицательное число) — в `worst_drop`.

*Подсказки (3): `dsc.hint('tree-shift')`*

In [ ]:
%%task ch08/01b-metric-tree/tree-shift
import pandas as pd

f = pd.read_csv("shop_funnel.csv")

### Ветка «чек» по кофейням · база

Средний чек снизился во всех кофейнях — проверьте это. По `shop_funnel.csv` посчитайте средний чек каждой кофейни (`revenue / checks`) в августе и сентябре и сохраните Series `check_pct` (индекс — `shop_id`) — изменение чека в процентах: (сентябрь / август − 1) × 100, 1 знак. В `worst_check` сохраните кофейню с самым большим падением чека.

*Подсказки (2): `dsc.hint('check-pct')`*

In [ ]:
%%task ch08/01b-metric-tree/check-pct
import pandas as pd

f = pd.read_csv("shop_funnel.csv")

### Сеть без KZN-1 · посложнее

Проверьте утверждение «почти вся потеря конверсии — одна кофейня». Уберите KZN-1 из `shop_funnel.csv` и разложите изменение выручки оставшихся пяти кофеен с августа на сентябрь в том же порядке «гости → конверсия → чек». Сохраните Series `parts_wo` (индекс `["гости", "конверсия", "чек"]`, рубли, целые) и `total_wo` — всё изменение выручки (целые рубли).

*Подсказки (2): `dsc.hint('tree-no-kzn')`*

In [ ]:
%%task ch08/01b-metric-tree/tree-no-kzn
import pandas as pd

f = pd.read_csv("shop_funnel.csv")

## Урок 4. Когда бюджет ограничен

Финансовый отдел выделил деньги только на 60 бонусов. Порог тут не поможет — нужен список «кому в первую очередь». Учимся ранжировать гостей по риску, читать кривую охвата и сравнивать модель с правилом при одном и том же бюджете.

Теория — на сайте курса: `ch08/02-budget.html`. Задачи «собери код из строк» (1) остались на сайте.

### Маска топ-k · разминка

Даны прогнозы шести гостей. Сохраните в `send` маску (Series `True/False` с тем же индексом) для двух гостей с наибольшим риском.

*Подсказки (2): `dsc.hint('top-mask')`*

In [ ]:
%%task ch08/02-budget/top-mask
import pandas as pd
risk = pd.Series([0.1, 0.8, 0.3, 0.9, 0.5, 0.2], index=["A", "B", "C", "D", "E", "F"])

### Точность и охват в топ-k · разминка

Посчитайте для модели при бюджете 100 бонусов точность в топ-k `prec_k` и охват `rec_k` (оба — 3 знака).

*Подсказки (2): `dsc.hint('at-k')`*

In [ ]:
%%task ch08/02-budget/at-k
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
# готово, как в теории: g, y, X, X_train, X_test, y_train, y_test, model, risk

### Прибыль по бюджетам · база

Посчитайте прибыль (400 ₽ за уходящего, −200 ₽ за лишний бонус) для бюджетов k = 20, 40, …, 300 (`range(20, 301, 20)`). Сохраните DataFrame `by_k` со столбцами `k` и `profit`, а в `best_k` — бюджет с наибольшей прибылью (при равенстве — меньший).

*Подсказки (3): `dsc.hint('profit-k')`*

In [ ]:
%%task ch08/02-budget/profit-k
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
# готово, как в теории: g, y, X, X_train, X_test, y_train, y_test, model, risk

### Случайная раздача · база

Посчитайте ожидаемую прибыль случайной раздачи k бонусов по формуле k × (rate × 400 − (1 − rate) × 200), где rate — доля уходящих в тесте. Сохраните `rnd_60` для k = 60 (округлите до целого) и `gain_60` — на сколько рублей модель при k = 60 лучше случайной раздачи (тоже до целого).

*Подсказки (2): `dsc.hint('random-base')`*

In [ ]:
%%task ch08/02-budget/random-base
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
# готово, как в теории: g, y, X, X_train, X_test, y_train, y_test, model, risk

### Лифт для финансиста · база

Сохраните долю уходящих в тесте `base` (3 знака), лифт модели для бюджетов 60 и 150 — `lift_60`, `lift_150` (2 знака), и лифт правила «давность визита» для 60 гостей с наибольшим `days_since_last` — `lift_rule_60` (2 знака).

*Подсказки (2): `dsc.hint('lift')`*

In [ ]:
%%task ch08/02-budget/lift
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
# готово, как в теории: g, y, X, X_train, X_test, y_train, y_test, model, risk

### Кривая охвата · база

Постройте кривую охвата для модели и диагональ случайного выбора на одном графике. Подпишите оси: X — `"доля гостей с бонусом"`, Y — `"доля пойманных уходящих"`. Сохраните в `at10` долю пойманных уходящих, если бонус получили первые 10 % гостей (60 человек), 3 знака.

*Подсказки (3): `dsc.hint('gains-plot')`*

In [ ]:
%%task ch08/02-budget/gains-plot
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
# готово, как в теории: g, y, X, X_train, X_test, y_train, y_test, model, risk

### Не тратить лишнее · база

Финансисты выделили 250 бонусов. Примените правило из подсказки урока: k = min(бюджет, число гостей с прогнозом выше 1/3). Сохраните `n_above` — число гостей выше порога, `k_use` — сколько бонусов отправить, `profit_all` — прибыль, если раздать все 250 по списку модели, `profit_cap` — если отправить только `k_use`, и `saved` — разницу.

*Подсказки (2): `dsc.hint('budget-cap')`*

In [ ]:
%%task ch08/02-budget/budget-cap
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
# готово, как в теории: g, y, X, X_train, X_test, y_train, y_test, model, risk, budget

### Бонус или звонок · посложнее

У отдела есть второй инструмент — звонок менеджера: стоит 500 ₽, удерживает 50 % уходящих (удержанный приносит 2400 ₽). Для звонка: выгода от уходящего 0.5 × 2400 − 500 = 700 ₽, лишний звонок −500 ₽. Бюджет на звонки — 30 000 ₽ (60 звонков), на бонусы — тоже 30 000 ₽ (150 бонусов). Получатели выбираются по одной и той же модели. Сохраните прибыль `calls_profit` (60 звонков самым рискованным) и `bonus_profit` (150 бонусов самым рискованным), а в `better` — `"звонки"` или `"бонусы"`.

*Подсказки (2): `dsc.hint('two-offers')`*

In [ ]:
%%task ch08/02-budget/two-offers
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
# готово, как в теории: g, y, X, X_train, X_test, y_train, y_test, model, risk

### Модель против давности при одном бюджете · со звёздочкой

Сравните модель и сортировку по давности визита (`days_since_last`, больше — рискованнее) при бюджетах 60, 100 и 150. Если у нескольких гостей одинаковая давность, берите тех, кто раньше в `X_test` — так работает `nlargest` по умолчанию. Сохраните DataFrame `cmp` со столбцами `k`, `model`, `days` (прибыль, +400 / −200) и в `gap_grows` — `True`, если разница «модель − давность» растёт с бюджетом (строго на каждом шаге).

*Подсказки (2): `dsc.hint('rule-gains')` · задача по желанию*

In [ ]:
%%task ch08/02-budget/rule-gains
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
# готово, как в теории: g, y, X, X_train, X_test, y_train, y_test, model, risk

## Урок 5. Риск: сколько можно потерять

«Кампания принесёт 34 тысячи» — это одно число по одной выборке гостей при одном допущении. Бизнесу нужно знать и разброс: как плохо может пойти и с какой вероятностью мы уйдём в минус. Считаем риск бутстрэпом и симуляцией.

Теория — на сайте курса: `ch08/03-risk.html`. Задачи «собери код из строк» (1) остались на сайте.

### Выборка с возвращением · разминка

Сгенерируйте генератором `np.random.default_rng(0)` 10 номеров строк от 0 до 9 (`rng.integers(0, 10, 10)`). Сохраните их в `idx`, а в `n_unique` — сколько разных номеров попало в выборку.

*Подсказки (2): `dsc.hint('resample')`*

In [ ]:
%%task ch08/03-risk/resample
import numpy as np
rng = np.random.default_rng(0)

### Сколько удержится · разминка

Бонус получили 120 уходящих гостей, каждый удерживается с вероятностью 0.25. Сгенерируйте 10 000 повторов числа удержанных: `rng.binomial(120, 0.25, size=10_000)` с `rng = np.random.default_rng(1)`. Сохраните в `p_low` долю повторов, где удержалось меньше 20 гостей (3 знака).

*Подсказки (2): `dsc.hint('binom')`*

In [ ]:
%%task ch08/03-risk/binom
import numpy as np
rng = np.random.default_rng(1)

### Интервал прибыли · база

Бутстрэпом (1000 повторов, `rng = np.random.default_rng(0)`, номера строк `rng.integers(0, len(yv), len(yv))`) оцените ожидаемую прибыль стратегии «бонус при прогнозе > 0.5»: +400 ₽ за уходящего, −200 ₽ за лишний бонус. Сохраните массив прибылей `profits`, границы 95 % интервала `low` и `high` (`np.percentile`, округлите до целого) и долю убыточных повторов `p_loss`.

*Подсказки (2): `dsc.hint('boot-profit')`*

In [ ]:
%%task ch08/03-risk/boot-profit
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
# готово, как в теории: g, y, X, X_train, X_test, y_train, y_test, model, send, yv

### Разорванные пары · база

Повторите бутстрэп прибыли для порога 1/3 (1000 повторов, `rng = np.random.default_rng(0)`) двумя способами. Правильный: один набор номеров `i` для `send` и `yv`. Ошибочный: `i` для `send` и отдельный `j = rng.integers(...)` для `yv` (в ошибочном цикле генерируйте сначала `i`, потом `j`; для каждого способа — свой новый генератор с seed 0). Сохраните средние прибыли `mean_ok` и `mean_broken` (округлите до целого) и `p_loss_broken` — долю убыточных повторов в ошибочном способе.

*Подсказки (2): `dsc.hint('broken-pairs')`*

In [ ]:
%%task ch08/03-risk/broken-pairs
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
# готово, как в теории: g, y, X, X_train, X_test, y_train, y_test, model, send, yv

### Риск пилота · база

Оцените риск пилота на 100 гостях: в каждом из 2000 повторов выберите 100 строк теста с возвращением (`rng.integers(0, len(yv), 100)`), посчитайте число уходящих с бонусом `tp`, число удержанных `rng.binomial(tp, 0.25)` и прибыль 2400 × удержанные − 200 × бонусы. Генератор — `np.random.default_rng(3)`, бонус — при прогнозе > 1/3. Проверка сравнивает с допуском, поэтому порядок вызовов генератора не важен; чтобы совпасть с разбором до рубля, вызывайте сначала `integers`, потом `binomial`. Сохраните `mean_pilot` (среднее, до целого) и `p_loss` (доля повторов с прибылью < 0, 3 знака).

*Подсказки (2): `dsc.hint('pilot')`*

In [ ]:
%%task ch08/03-risk/pilot
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
# готово, как в теории: g, y, X, X_train, X_test, y_train, y_test, model, send, yv

### Формула против симуляции · база

Проверьте прикидку из теории. Сгенерируйте 10 000 повторов числа удержанных из 113 уходящих (`rng = np.random.default_rng(3)`, `rng.binomial(113, 0.25, size=10_000)`). Сохраните `sd_sim` — стандартное отклонение симуляции, `sd_formula` — √(n × p × (1 − p)) (оба — 2 знака), и `sd_rub` — разброс прибыли в рублях по формуле (`sd_formula` × 2400, округлите до целого).

*Подсказки (2): `dsc.hint('binom-sd')`*

In [ ]:
%%task ch08/03-risk/binom-sd
import numpy as np
rng = np.random.default_rng(3)
n, p = 113, 0.25

### Порог по плохому сценарию · база

Сравните бутстрэпом пороги 0.2, 0.35 и 0.5 (ожидаемая прибыль +400 / −200, 1000 повторов). Пороги и генератор здесь другие, чем в таблице «Риск разных стратегий» (там 1/3), поэтому числа с таблицей не совпадут — сравнивайте пороги между собой. Для каждого порога создавайте **новый** генератор `np.random.default_rng(0)`, чтобы все пороги проверялись на одних и тех же «месяцах». Сохраните DataFrame `risk` со столбцами `threshold`, `mean`, `p5` (среднее и 5-й процентиль, до целого) и в `safest` — порог с наибольшим `p5`.

*Подсказки (2): `dsc.hint('compare-risk')`*

In [ ]:
%%task ch08/03-risk/compare-risk
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
# готово, как в теории: g, y, X, X_train, X_test, y_train, y_test, model, p, yv

### Неточное допущение · посложнее

Добавьте неуверенность в эффекте: в каждом из 2000 повторов сначала берите удержание `r = rng.uniform(0.15, 0.35)`, затем строки теста с возвращением (`rng.integers(0, len(yv), len(yv))`), затем удержанных `rng.binomial(tp, r)`. Генератор — `np.random.default_rng(4)` (порядок вызовов не важен — проверка с допуском), бонус при прогнозе > 1/3, прибыль 2400 × удержанные − 200 × бонусы. Сохраните `p_loss` (3 знака) и `launch` — `True`, если вероятность убытка меньше 2.5 %.

*Подсказки (2): `dsc.hint('unsure-retain')`*

In [ ]:
%%task ch08/03-risk/unsure-retain
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
# готово, как в теории: g, y, X, X_train, X_test, y_train, y_test, model, send, yv

### При каком удержании риск приемлем · со звёздочкой

Найдите наименьшее удержание из `np.arange(0.10, 0.31, 0.02).round(2)`, при котором вероятность убытка кампании (порог 1/3, 1000 повторов: строки с возвращением + `rng.binomial(tp, r)`, прибыль 2400 × удержанные − 200 × бонусы) меньше 2.5 %. Для каждого значения удержания создавайте новый генератор `np.random.default_rng(5)`. Сохраните найденное значение в `min_retain`.

*Подсказки (2): `dsc.hint('break-retain')` · задача по желанию*

In [ ]:
%%task ch08/03-risk/break-retain
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
# готово, как в теории: g, y, X, X_train, X_test, y_train, y_test, model, send, yv

## Урок 6. Модель в работе: мониторинг

Модель приоритизации заявок из проекта главы 7 запустили в мае. Через три месяца менеджеры жалуются: «модель перестала выделять хорошие заявки». Учимся следить за моделью после запуска — по входам, выходам и качеству — находить поломку раньше, чем её заметит бизнес, и решать, когда модель переобучать. Качество по сегментам и отчёт о метрике — в следующем уроке.

Теория — на сайте курса: `ch08/04-monitoring.html`. Задачи «собери код из строк» (1) остались на сайте.

### Месяц заявки · разминка

Добавьте в таблицу `d` столбец `month` — номер месяца из `created`, и сохраните в `per_month` число заявок по месяцам (Series, индекс — месяц).

*Подсказки (2): `dsc.hint('month-col')`*

In [ ]:
%%task ch08/04-monitoring/month-col
import pandas as pd
d = pd.read_csv("catering_leads.csv", parse_dates=["created"])

### Доля пропусков по месяцам · разминка

Посчитайте долю пропусков `budget_pp` по месяцам (3 знака) и сохраните Series `miss`. В `max_month` — месяц с наибольшей долей.

*Подсказки (2): `dsc.hint('ch08/04-monitoring/miss-share')`*

In [ ]:
%%task ch08/04-monitoring/miss-share
import pandas as pd
d = pd.read_csv("catering_leads.csv", parse_dates=["created"])
d["month"] = d["created"].dt.month

### Качество по месяцам · база

В заготовке модель обучена на январе–апреле (переменные `d`, `num`, `cat`, `model` уже есть). Посчитайте для мая–сентября DataFrame `quality` со столбцами `month`, `fact` (конверсия), `predicted` (средний прогноз), `auc` — все дробные значения с 3 знаками. В `worst_month` сохраните месяц с наименьшим AUC.

*Подсказки (2): `dsc.hint('monthly-auc')`*

In [ ]:
%%task ch08/04-monitoring/monthly-auc
# уже есть: d (со столбцом month), num, cat, model (обучена на январе–апреле)
from sklearn.metrics import roc_auc_score

### Новые категории · база

Смоделируйте переименование из «Разберём вместе»: в копии `live` замените в сентябре канал «сайт» на «website». Сохраните словарь `new_cats` — для каждого месяца с мая список каналов (отсортированный), которых не было в обучении (`train`), число переименованных заявок `n_renamed` и средний прогноз для них до (`pred_before`, по `d`) и после переименования (`pred_after`, по `live`), оба — 3 знака.

*Подсказки (3): `dsc.hint('new-category')`*

In [ ]:
%%task ch08/04-monitoring/new-category
# уже есть: d, train (январь–апрель), num, cat, model
live = d.copy()

### Тревога по медианам · база

В заготовке `live` — данные, где с августа время ответа записано в минутах. Для каждого месяца с мая посчитайте отношение медианы признака к медиане за январь–апрель по признакам `guests`, `budget_pp`, `days_to_event`, `response_hours`. Сохраните DataFrame `ratio` (строки — месяцы, столбцы — признаки) и список `alarms` — пары `(месяц, признак)`, где отношение меньше 0.5 или больше 2, в порядке месяцев, а внутри месяца — в порядке признаков.

*Подсказки (2): `dsc.hint('median-alarm')`*

In [ ]:
%%task ch08/04-monitoring/median-alarm
import pandas as pd
d = pd.read_csv("catering_leads.csv", parse_dates=["created"])
d["month"] = d["created"].dt.month
live = d.copy()
aug = live["month"] >= 8
live.loc[aug, "response_hours"] = (live.loc[aug, "response_hours"] * 60).round()
cols = ["guests", "budget_pp", "days_to_event", "response_hours"]

### Слепое пятно AUC · база

Покажите, что AUC не видит общего сдвига. Возьмите сентябрьские заявки из `d` (без сбоя), прогнозы `p` модели и сдвиньте все логиты на −2: `p2 = 1 / (1 + np.exp(-(np.log(p / (1 - p)) - 2)))`. Логит — это `log(p / (1 − p))`, логарифм шансов (глава 6, урок «Логистическая регрессия и вероятности»); обратно его переводит сигмоида. Вычесть 2 из логита — значит уменьшить шансы каждой заявки в e² ≈ 7 раз: все вероятности падают, но порядок заявок не меняется. Так выглядит общий сдвиг, когда поломка задевает всех одинаково. Сохраните `auc_p`, `auc_p2` (3 знака), долю заявок выше 0.3 `above_p`, `above_p2` и средние прогнозы `mean_p`, `mean_p2` (все — 3 знака).

*Подсказки (2): `dsc.hint('auc-blind')`*

In [ ]:
%%task ch08/04-monitoring/auc-blind
import numpy as np
from sklearn.metrics import roc_auc_score
# уже есть: d, num, cat, model
sep = d[d["month"] == 9]

### Тревога по выходу модели · база

На данных `live` (минуты с августа) посчитайте для каждого месяца с мая долю заявок с прогнозом выше 0.3. Сравните с той же долей на обучающих данных (`train_share`, 3 знака). Сохраните Series `above` (индекс — месяц, 3 знака) и список `alarm_months` — месяцы, где доля отличается от обучающей больше чем вдвое (меньше половины или больше двух).

*Подсказки (2): `dsc.hint('output-alarm')`*

In [ ]:
%%task ch08/04-monitoring/output-alarm
# уже есть: d, live (минуты с августа), train (январь–апрель), num, cat, model

### Починить и переобучить · посложнее

Сбой нашли: с августа минуты переводим обратно в часы. Сравните на августе–сентябре (данные `d`, уже без сбоя) две модели: `old` — обучена на январе–апреле (есть в заготовке как `model`), `new` — такая же, обученная на январе–июле (создайте функцией `make_model()`). Сохраните `auc_old`, `auc_new` (3 знака) и `use_new` — `True`, если новая модель лучше.

*Подсказки (2): `dsc.hint('fix-retrain')`*

In [ ]:
%%task ch08/04-monitoring/fix-retrain
# уже есть: d, num, cat, make_model() — новый необученный конвейер, model — обучена на январе–апреле
from sklearn.metrics import roc_auc_score

### Индекс стабильности · со звёздочкой

Напишите функцию `psi(a, b, bins)`: доли значений `a` и `b` по интервалам `pd.cut(..., bins)` (`value_counts(normalize=True, sort=False)`), к каждой доле прибавьте 0.0001 (чтобы не делить на ноль), и верните сумму (b − a) × ln(b / a). Посчитайте PSI `response_hours` между январём–апрелем и каждым месяцем с мая на данных `live` (минуты с августа), интервалы `[0, 1, 2, 4, 8, 100000]`. Сохраните Series `psi_by_month` (3 знака) и список `serious` — месяцы с PSI больше 0.25.

*Подсказки (2): `dsc.hint('psi')` · задача по желанию*

In [ ]:
%%task ch08/04-monitoring/psi
import numpy as np
import pandas as pd
d = pd.read_csv("catering_leads.csv", parse_dates=["created"])
d["month"] = d["created"].dt.month
live = d.copy()
aug = live["month"] >= 8
live.loc[aug, "response_hours"] = (live.loc[aug, "response_hours"] * 60).round()
bins = [0, 1, 2, 4, 8, 100000]

## Урок 7. Качество по сегментам и отчёт о метрике

Одно число качества на всю сеть прячет слабые места: модель может хорошо ранжировать заявки в целом и при этом путаться внутри каждого канала. Считаем качество по сегментам с доверительными интервалами и учимся писать о метрике так, чтобы из отчёта нельзя было сделать неправильный вывод.

Теория — на сайте курса: `ch08/04b-segments.html`.

### Качество по городам · база

Постройте для заявок мая–сентября таблицу `by_city` по городам (индекс — город) со столбцами `leads`, `deals`, `fact`, `pred`, `auc` (как в теории, но по `city`), округлите до 3 знаков. Затем сохраните в `gap_city` город, где модель сильнее всего **занижает** конверсию (наибольшая разность `fact − pred`), а в `gap` — эту разность (3 знака).

*Подсказки (3): `dsc.hint('seg-table')`*

In [ ]:
%%task ch08/04b-segments/seg-table
import pandas as pd
from sklearn.pipeline import make_pipeline
from sklearn.compose import make_column_transformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

d = pd.read_csv("catering_leads.csv", parse_dates=["created"])
d["month"] = d["created"].dt.month
num = ["guests", "budget_pp", "days_to_event", "response_hours", "repeat_client", "tasting"]
cat = ["channel", "event_type", "city", "company_size", "manager"]
prep = make_column_transformer(
    (make_pipeline(SimpleImputer(strategy="median", add_indicator=True), StandardScaler()), num),
    (make_pipeline(SimpleImputer(strategy="constant", fill_value="не указан"),
                   OneHotEncoder(handle_unknown="ignore")), cat),
)
model = make_pipeline(prep, LogisticRegression(max_iter=1000))
train = d[d["month"] <= 4]
model.fit(train[num + cat], train["converted"])
live = d[d["month"] >= 5].copy()
live["p"] = model.predict_proba(live[num + cat])[:, 1]

### Отчёт о метрике · посложнее

Напишите функцию `metric_report(y, p, n_boot=200, seed=0)`, которая возвращает словарь с ключами `n`, `positives`, `auc`, `ci_low`, `ci_high`, `pred`, `fact` (числа, 3 знака где дробные). Интервал — бутстрэп, как в теории: `rng = np.random.default_rng(seed)`, `n_boot` раз выбрать номера строк `rng.integers(0, len(y), len(y))`, пропустить выборки с одним классом, взять процентили 2.5 и 97.5. Примените её ко всем заявкам мая–сентября (`rep_all`) и к заявкам канала «соцсети» (`rep_social`). В `wider` сохраните, во сколько раз интервал соцсетей шире общего (`(ci_high − ci_low)` соцсетей / то же общее, 1 знак).

*Подсказки (3): `dsc.hint('metric-report')`*

In [ ]:
%%task ch08/04b-segments/metric-report
import numpy as np
import pandas as pd
from sklearn.pipeline import make_pipeline
from sklearn.compose import make_column_transformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

d = pd.read_csv("catering_leads.csv", parse_dates=["created"])
d["month"] = d["created"].dt.month
num = ["guests", "budget_pp", "days_to_event", "response_hours", "repeat_client", "tasting"]
cat = ["channel", "event_type", "city", "company_size", "manager"]
prep = make_column_transformer(
    (make_pipeline(SimpleImputer(strategy="median", add_indicator=True), StandardScaler()), num),
    (make_pipeline(SimpleImputer(strategy="constant", fill_value="не указан"),
                   OneHotEncoder(handle_unknown="ignore")), cat),
)
model = make_pipeline(prep, LogisticRegression(max_iter=1000))
train = d[d["month"] <= 4]
model.fit(train[num + cat], train["converted"])
live = d[d["month"] >= 5].copy()
live["p"] = model.predict_proba(live[num + cat])[:, 1]

def metric_report(y, p, n_boot=200, seed=0):
    ...

### Игрушечный пример своими руками · разминка

Восемь заявок из двух каналов уже лежат в таблице `t`. Посчитайте `auc_all` — AUC по всем восьми заявкам и `auc_seg` — Series с AUC внутри каждого канала (индекс — канал). Округлите до 2 знаков. В `higher` сохраните `True`, если общий AUC выше AUC в каждом из каналов.

*Подсказки (2): `dsc.hint('seg-toy')`*

In [ ]:
%%task ch08/04b-segments/seg-toy
import pandas as pd
from sklearn.metrics import roc_auc_score

t = pd.DataFrame({
    "channel": ["рекомендация"] * 4 + ["соцсети"] * 4,
    "p": [0.8, 0.7, 0.6, 0.5, 0.4, 0.3, 0.2, 0.1],
    "deal": [1, 1, 0, 1, 0, 1, 0, 0],
})

### Модель против простого правила по каналам · база

Сравните модель с правилом «повторный клиент — первым» внутри каждого канала. Заявки мая–сентября с прогнозом `p` уже лежат в `live`. Сохраните `rule_auc` — AUC правила по всем заявкам (`repeat_client` вместо прогноза, 3 знака), Series `gain` (индекс — канал) — на сколько AUC модели выше AUC правила внутри канала (3 знака), и в `least` — канал, где модель добавляет к правилу меньше всего.

*Подсказки (2): `dsc.hint('rule-base')`*

In [ ]:
%%task ch08/04b-segments/rule-base
import pandas as pd
from sklearn.metrics import roc_auc_score
# готово, как в теории: d, model, live (заявки мая–сентября, столбец p — прогноз)

### Отчёт словами · база

Метрику мало посчитать — её надо написать так, чтобы из текста нельзя было сделать неверный вывод. Соберите f-строку `report` для канала «соцсети» по образцу из теории. В ней должны быть: число заявок и договоров, AUC (3 знака), 95 % интервал (2 знака, функция `auc_ci` из теории), средний прогноз и фактическая конверсия (3 знака). Например: `"Соцсети: AUC 0.852 (95 % ДИ 0.79–0.92) на 324 заявках, 22 договора. Прогноз 0.025 при факте 0.068."` — но числа должны браться из переменных, а не быть вписаны руками.

*Подсказки (2): `dsc.hint('report-text')`*

In [ ]:
%%task ch08/04b-segments/report-text
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
# готово, как в теории: d, model, live (заявки мая–сентября, столбец p — прогноз)

def auc_ci(y, p, n_boot=200, seed=0):
    y, p = np.asarray(y), np.asarray(p)
    rng = np.random.default_rng(seed)
    aucs = []
    for _ in range(n_boot):
        i = rng.integers(0, len(y), len(y))
        if y[i].min() == y[i].max():
            continue
        aucs.append(roc_auc_score(y[i], p[i]))
    return np.percentile(aucs, [2.5, 97.5])

soc = live[live["channel"] == "соцсети"]

## Урок 8. Объяснить модель бизнесу

Директор по лояльности спрашивает: «Почему модель считает, что этот гость уйдёт? И от чего уход вообще зависит?» Учимся отвечать на оба вопроса — о модели в целом и об одном гостевом прогнозе — и не выдавать связь за причину. Справедливость модели по группам гостей — в следующем уроке.

Теория — на сайте курса: `ch08/05-explain.html`. Задачи «собери код из строк» (1) остались на сайте.

### Одна точка кривой · разминка

Поставьте всем гостям теста `days_since_last = 30` и сохраните средний прогноз леса в `risk_30` (3 знака). Исходную таблицу `X_test` не меняйте.

*Подсказки (2): `dsc.hint('pd-one')`*

In [ ]:
%%task ch08/05-explain/pd-one
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

g = pd.read_csv("guests_churn.csv")
y = g["churned"]
X = pd.get_dummies(g.drop(columns=["guest_id", "churned", "card_blocked"]), dtype=int).fillna({"avg_check": 0})
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
forest = RandomForestClassifier(n_estimators=100, min_samples_leaf=5, random_state=0).fit(X_train, y_train)

### Вклад на калькуляторе · разминка

Повторите таблицу из «Разберём вместе» для гостя с 12 визитами за квартал (`visits_90d`): среднее 6.36, разброс 5.09, вес −0.74. Сохраните `z` и `contrib` (2 знака) и `direction` — `"повышает"` или `"снижает"` риск.

*Подсказки (2): `dsc.hint('contrib-hand')`*

In [ ]:
%%task ch08/05-explain/contrib-hand
value, mean, scale, coef = 12, 6.36, 5.09, -0.74

### Новички · база

С помощью частичной зависимости по `tenure_months` для сетки `[1, 2, 3, 4, 6, 12, 24]` сохраните список средних прогнозов `pd_tenure` (3 знака) и в `newbie_ratio` — во сколько раз прогноз при 2 месяцах больше прогноза при 12 (2 знака).

*Подсказки (2): `dsc.hint('pd-tenure')`*

In [ ]:
%%task ch08/05-explain/pd-tenure
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

g = pd.read_csv("guests_churn.csv")
y = g["churned"]
X = pd.get_dummies(g.drop(columns=["guest_id", "churned", "card_blocked"]), dtype=int).fillna({"avg_check": 0})
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
forest = RandomForestClassifier(n_estimators=100, min_samples_leaf=5, random_state=0).fit(X_train, y_train)
grid = [1, 2, 3, 4, 6, 12, 24]

### Веса для людей · база

Обучите `make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))` и сохраните Series весов `w` с индексом — названиями признаков. В `top_up` сохраните список трёх признаков с самыми большими положительными весами (по убыванию), а в `top_down` — трёх с самыми отрицательными (от самого отрицательного).

*Подсказки (2): `dsc.hint('ch08/05-explain/coefs')`*

In [ ]:
%%task ch08/05-explain/coefs
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

g = pd.read_csv("guests_churn.csv")
y = g["churned"]
X = pd.get_dummies(g.drop(columns=["guest_id", "churned", "card_blocked"]), dtype=int).fillna({"avg_check": 0})
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

### Карточка гостя · база

Для гостя с индексом 1605 (G-11606) посчитайте вклады признаков в оценку логистической регрессии: (значение − среднее) / стандартное отклонение × вес. Сохраните Series `contrib` (индекс — признаки) и строку `card` вида `"риск 98%: complaints, days_since_last, visits_90d"` — риск в целых процентах и три признака с наибольшим положительным вкладом через запятую.

*Подсказки (3): `dsc.hint('guest-card')`*

In [ ]:
%%task ch08/05-explain/guest-card
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

g = pd.read_csv("guests_churn.csv")
y = g["churned"]
X = pd.get_dummies(g.drop(columns=["guest_id", "churned", "card_blocked"]), dtype=int).fillna({"avg_check": 0})
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
logit = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(X_train, y_train)

### Самый рискованный гость · база

Найдите в тесте гостя с наибольшим прогнозом логистической регрессии. Сохраните его индекс `top_idx`, риск `top_risk` (3 знака), Series вкладов `contrib` и `top3` — список трёх признаков с наибольшим вкладом. В `days_contrib` — вклад `days_since_last` (2 знака).

*Подсказки (2): `dsc.hint('top-guest')`*

In [ ]:
%%task ch08/05-explain/top-guest
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

g = pd.read_csv("guests_churn.csv")
y = g["churned"]
X = pd.get_dummies(g.drop(columns=["guest_id", "churned", "card_blocked"]), dtype=int).fillna({"avg_check": 0})
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
logit = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(X_train, y_train)

### Вклады складываются в прогноз · посложнее

Проверьте, что объяснение «честное»: свободный член плюс сумма вкладов гостя 1605 равны оценке z, а сигмоида от неё — прогнозу модели. Сохраните `z` (свободный член + сумма вкладов), `p_manual` = 1 / (1 + e^(−z)) и `p_model` — прогноз `predict_proba` (оба 4 знака).

*Подсказки (2): `dsc.hint('sum-check')`*

In [ ]:
%%task ch08/05-explain/sum-check
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

g = pd.read_csv("guests_churn.csv")
y = g["churned"]
X = pd.get_dummies(g.drop(columns=["guest_id", "churned", "card_blocked"]), dtype=int).fillna({"avg_check": 0})
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
logit = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(X_train, y_train)
scaler, lr = logit[0], logit[-1]
contrib = (X_test.loc[1605] - scaler.mean_) / scaler.scale_ * lr.coef_[0]

### Что если поставить приложение · посложнее

Маркетинг хочет уговаривать гостей поставить приложение. Возьмите гостей теста без приложения с прогнозом леса выше 0.3, поставьте им `app = 1` и посчитайте средний прогноз до и после. Сохраните `n_group`, `before`, `after` (3 знака) и `drop_pp` — снижение в процентных пунктах (1 знак).

*Подсказки (2): `dsc.hint('what-if-app')`*

In [ ]:
%%task ch08/05-explain/what-if-app
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

g = pd.read_csv("guests_churn.csv")
y = g["churned"]
X = pd.get_dummies(g.drop(columns=["guest_id", "churned", "card_blocked"]), dtype=int).fillna({"avg_check": 0})
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
forest = RandomForestClassifier(n_estimators=100, min_samples_leaf=5, random_state=0).fit(X_train, y_train)
p = pd.Series(forest.predict_proba(X_test)[:, 1], index=X_test.index)

### Две кривые · со звёздочкой

Постройте частичную зависимость по давности визита (`grid = [0, 7, 14, 21, 30, 45, 60, 90]`) отдельно для новичков (`tenure_months <= 3`) и остальных гостей теста — две линии на одном графике с легендой «новички» и «остальные», ось X — `"дней с последнего визита"`. Сохраните списки `pd_new` и `pd_old` (3 знака) и в `max_gap_day` — значение сетки, при котором разница «новички − остальные» наибольшая.

*Подсказки (2): `dsc.hint('pd-two')` · задача по желанию*

In [ ]:
%%task ch08/05-explain/pd-two
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

g = pd.read_csv("guests_churn.csv")
y = g["churned"]
X = pd.get_dummies(g.drop(columns=["guest_id", "churned", "card_blocked"]), dtype=int).fillna({"avg_check": 0})
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
forest = RandomForestClassifier(n_estimators=100, min_samples_leaf=5, random_state=0).fit(X_train, y_train)
grid = [0, 7, 14, 21, 30, 45, 60, 90]

## Урок 9. Этика данных и справедливость модели

Модель, которая хорошо работает в среднем, может систематически недодавать одной группе гостей — даже если возраста или пола среди признаков нет. Разбираемся, какие данные брать нельзя, как проверить справедливость по группам на честных прогнозах и как найти признаки-заместители.

Теория — на сайте курса: `ch08/05b-fairness.html`.

### Справедливость при другом пороге · база

Маркетинг предлагает снизить порог до 0.25, «чтобы старшие гости тоже получали бонусы». Проверьте. По прогнозам вне выборки `p` (уже посчитаны в заготовке) отправьте бонус при `p > 0.25` и соберите таблицу `fair25` по `age_group` со столбцами `selected` (доля получивших бонус) и `recall` (доля получивших среди уходящих), 3 знака. Сохраните в `gap25` разницу между наибольшей и наименьшей полнотой групп (3 знака), в `extra_bonuses` — на сколько бонусов больше, чем при пороге 1/3, а в `fixes_gap` — ответ маркетингу: `True`, если при пороге 0.25 разрыв полноты стал меньше 0.1 (то есть перекос почти исчез).

*Подсказки (3): `dsc.hint('fair-recall')`*

In [ ]:
%%task ch08/05b-fairness/fair-recall
import pandas as pd
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier

g = pd.read_csv("guests_churn.csv").merge(pd.read_csv("guests_age.csv"), on="guest_id", validate="one_to_one")
y = g["churned"]
X = pd.get_dummies(g.drop(columns=["guest_id", "churned", "card_blocked", "age_group"]), dtype=int).fillna({"avg_check": 0})
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
forest = RandomForestClassifier(n_estimators=100, min_samples_leaf=5, random_state=0)
p = cross_val_predict(forest, X, y, cv=cv, method="predict_proba")[:, 1]

### Заместители возраста · посложнее

Найдите, какой признак модели сильнее всего «выдаёт» группу 50+. Для каждого столбца `X` посчитайте ROC-AUC восстановления `old` по одному этому признаку; направление неважно, поэтому берите `max(auc, 1 − auc)`. Сохраните Series `single` (индекс — названия столбцов, 3 знака) и имя лучшего в `top_proxy`. Затем сравните качество модели оттока с возрастом и без: средний ROC-AUC по 5 фолдам (`cross_val_score` с `cv` и `forest` из заготовки) для `X` — в `auc_without`, для `X` плюс one-hot возраста (`pd.get_dummies(g["age_group"], prefix="age", dtype=int)`) — в `auc_with`, оба 3 знака.

*Подсказки (3): `dsc.hint('proxy-check')`*

In [ ]:
%%task ch08/05b-fairness/proxy-check
import pandas as pd
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

g = pd.read_csv("guests_churn.csv").merge(pd.read_csv("guests_age.csv"), on="guest_id", validate="one_to_one")
y = g["churned"]
X = pd.get_dummies(g.drop(columns=["guest_id", "churned", "card_blocked", "age_group"]), dtype=int).fillna({"avg_check": 0})
old = (g["age_group"] == "50+").astype(int)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
forest = RandomForestClassifier(n_estimators=100, min_samples_leaf=5, random_state=0)

### Три метрики на двенадцати гостях · разминка

В таблице `t` двенадцать гостей двух групп: ушёл ли гость (`churned`) и отправила ли ему модель бонус (`send`). Посчитайте по группам `selected` — долю получивших бонус среди всех гостей группы, `recall` — долю получивших среди уходящих и `precision` — долю уходящих среди получивших. Сохраните их тремя Series (индекс — группа, 2 знака).

*Подсказки (2): `dsc.hint('fair-toy')`*

In [ ]:
%%task ch08/05b-fairness/fair-toy
import pandas as pd

t = pd.DataFrame({
    "group":   ["A"] * 6 + ["B"] * 6,
    "churned": [1, 1, 1, 0, 0, 0,   1, 1, 1, 0, 0, 0],
    "send":    [1, 1, 1, 0, 0, 0,   1, 0, 0, 1, 1, 0],
})

### Кого модель пропускает · база

Проверьте утверждение из теории «из 120 пропущенных уходящих 48 — старшие, хотя их всего 18 % в базе». Прогнозы вне выборки уже посчитаны, бонус — `g["send"]` (порог 1/3). Сохраните `missed_n` — сколько уходящих не получили бонус, Series `missed_share` — доли групп среди них (индекс — `age_group`, 3 знака) и `base_old` — долю гостей 50+ во всей базе (3 знака).

*Подсказки (2): `dsc.hint('missed')`*

In [ ]:
%%task ch08/05b-fairness/missed
import pandas as pd
# готово, как в теории: g (с age_group и send), p — прогнозы вне выборки

### Тихий уход · база

Почему модель пропускает старших? Сравните уходящих гостей разных групп. Возьмите только `churned == 1` и сохраните DataFrame `quiet` (индекс — `age_group`) со столбцами `med_days` — медиана `days_since_last` и `complained` — доля гостей с жалобой (`complaints > 0`, 3 знака). В `quietest` — группу с наименьшей долей жалоб. Модель здесь не нужна, только `guests_churn.csv` и `guests_age.csv`.

*Подсказки (2): `dsc.hint('quiet-churn')`*

In [ ]:
%%task ch08/05b-fairness/quiet-churn
import pandas as pd

g = pd.read_csv("guests_churn.csv").merge(pd.read_csv("guests_age.csv"), on="guest_id", validate="one_to_one")

## Урок 10. Проект: где поставить кофейные автоматы

Проект главы (итоговый проект первой части курса про ML — следующий урок). «Зерно» выходит в новый формат — кофейные автоматы в бизнес-центрах, вузах и на вокзалах. Есть деньги на 40 автоматов в одном из трёх городов. Нужно выбрать город и места так, чтобы заработать — и не рискнуть деньгами сверх допустимого.

Теория — на сайте курса: `ch08/06-project.html`.

### Шаг 1. Сколько мест окупается · база

Посчитайте по городам число мест, среднюю прибыль (1 знак) и долю мест, где прибыль больше стоимости автомата (140 тыс. ₽), 3 знака. Сохраните DataFrame `overview` (индекс — город, столбцы `sites`, `mean_profit`, `pays_off`) и в `best_avg_city` — город с наибольшей средней прибылью.

*Подсказки (2): `dsc.hint('ch08/06-project/look')`*

In [ ]:
%%task ch08/06-project/look
import pandas as pd
s = pd.read_csv("sites.csv")

### Шаг 2. Признаки · база

Напишите функцию `prep(part)`, которая из таблицы мест делает матрицу признаков: столбцы `place`, `traffic`, `offices`, `competitors`, `rent_k`, `hours24`, тип места — через `pd.get_dummies(..., dtype=int)`, а `traffic` заменён натуральным логарифмом (`np.log`). Сохраните `X_perm = prep(...)` для мест Перми.

*Подсказки (2): `dsc.hint('prep')`*

In [ ]:
%%task ch08/06-project/prep
import numpy as np
import pandas as pd
s = pd.read_csv("sites.csv")

### Шаг 3. Модели по городам · база

Для каждого города разделите места на обучение и тест (`train_test_split(X, y, test_size=0.25, random_state=42)`), обучите `LinearRegression` и посчитайте на тесте R² (3 знака) и RMSE (1 знак). Сохраните DataFrame `quality` (индекс — город, столбцы `r2`, `rmse`) и в `most_predictable` — город с наибольшим R².

*Подсказки (2): `dsc.hint('ch08/06-project/models')`*

In [ ]:
%%task ch08/06-project/models
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error

s = pd.read_csv("sites.csv")

def prep(part):
    X = pd.get_dummies(part[["place", "traffic", "offices", "competitors", "rent_k", "hours24"]], dtype=int)
    X["traffic"] = np.log(X["traffic"])
    return X

### Шаг 4. Сорок лучших мест · база

В заготовке для каждого города уже есть прогнозы и факты на тесте: `fits[city] = (pred, fact)` — две Series с одинаковым индексом. Для каждого города выберите 40 мест с наибольшим прогнозом и посчитайте прибыль: сумма фактической `profit_k` по выбранным минус 40 × 140. Для сравнения посчитайте ожидаемую прибыль 40 случайных мест теста: 40 × средняя `profit_k` теста − 40 × 140. Сохраните DataFrame `top` (индекс — город, столбцы `model`, `random`, оба — до целого).

*Подсказки (2): `dsc.hint('top40')`*

In [ ]:
%%task ch08/06-project/top40
# уже есть: s, prep(part), fits[city] = (pred, fact) — прогноз и факт на тестовых местах города
COST, K = 140, 40

### Шаг 4б. Обещание против факта · база

Проверьте проклятие победителя из «Разберём вместе» на тесте. Для каждого города возьмите 40 мест с наибольшим прогнозом и посчитайте `promised` — прибыль по **прогнозам** (сумма прогнозов − 40 × 140) и `actual` — по **фактам**. Сохраните DataFrame `promise` (индекс — город, столбцы `promised`, `actual`, `gap = actual − promised`, все — до целого) и `overpromised` — список городов, где модель пообещала больше, чем получилось.

*Подсказки (2): `dsc.hint('promise')`*

In [ ]:
%%task ch08/06-project/promise
# уже есть: s, prep(part), fits[city] = (pred, fact)
COST, K = 140, 40

### Шаг 5. Риск по городам · посложнее

Напишите функцию `boot_city(pred, fact)`: генератор `np.random.default_rng(0)`, 1000 повторов; в каждом — 300 номеров мест с возвращением `rng.integers(0, len(fact), 300)`, из них 40 с наибольшим прогнозом (`np.argsort(-p[i])[:40]` по массиву прогнозов), прибыль = сумма фактов выбранных − 40 × 140. Функция возвращает массив из 1000 прибылей. Соберите DataFrame `risk` (индекс — город, столбцы `mean`, `low`, `high`, `p_loss`): среднее и 2.5-й / 97.5-й процентили (до целого), доля убыточных повторов (3 знака).

*Подсказки (2): `dsc.hint('ch08/06-project/boot')`*

In [ ]:
%%task ch08/06-project/boot
# уже есть: s, prep(part), fits[city] = (pred, fact)
import numpy as np
COST, K, SURVEY = 140, 40, 300

### Шаг 6. Решение · база

Примените правило совета директоров к таблице `risk` из прошлого шага (в заготовке она уже посчитана): оставьте города с вероятностью убытка не больше 0.025 и выберите среди них город с наибольшей средней прибылью. Сохраните список прошедших городов `passed` (в порядке индекса `risk`) и `choice` — выбранный город. В `cost_of_safety` сохраните, сколько средней прибыли (тыс. ₽) мы отдаём за безопасность: средняя у лидера по среднему минус средняя у выбранного города.

*Подсказки (2): `dsc.hint('ch08/06-project/decision')`*

In [ ]:
%%task ch08/06-project/decision
# уже есть: risk — таблица из шага 5 (mean, low, high, p_loss по городам)
print(risk)

### Шаг 6б. Какие места выбирает модель · база

Директору нужен не только город, но и понимание, какие места модель считает лучшими. Для Перми возьмите 40 лучших по прогнозу мест теста и сохраните: `bc_share` — долю бизнес-центров среди них (3 знака), `pays_top` и `pays_all` — долю окупаемых мест (`profit_k > 140`) среди выбранных и среди всех мест теста (3 знака), `traffic_top` и `traffic_all` — медианный трафик выбранных и всех мест теста, `comp_top` и `comp_all` — медианное число конкурентов рядом, `rent_top` и `rent_all` — медианную аренду (тыс. ₽).

*Подсказки (2): `dsc.hint('profile')`*

In [ ]:
%%task ch08/06-project/profile
# уже есть: s, prep(part), fits[city] = (pred, fact)
pred, fact = fits["Пермь"]

### Шаг 7. А если взять лес · со звёздочкой

Проверьте, не изменит ли решение более сложная модель. Повторите шаги 3–5 для Перми и Екатеринбурга с `RandomForestRegressor(n_estimators=100, min_samples_leaf=5, random_state=0)` вместо линейной регрессии (те же признаки и то же разбиение). Сохраните DataFrame `rf_risk` (индекс — эти два города, столбцы `r2` (3 знака), `mean` (до целого), `p_loss` (3 знака)) и `same_choice` — `True`, если по правилу совета (только эти два города) выбор тот же, что с линейной моделью (Пермь).

*Подсказки (2): `dsc.hint('forest')` · задача по желанию*

In [ ]:
%%task ch08/06-project/forest
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score

s = pd.read_csv("sites.csv")

def prep(part):
    X = pd.get_dummies(part[["place", "traffic", "offices", "competitors", "rent_k", "hours24"]], dtype=int)
    X["traffic"] = np.log(X["traffic"])
    return X

def boot_city(pred, fact):
    rng = np.random.default_rng(0)
    p, f = pred.values, fact.values
    out = []
    for _ in range(1000):
        i = rng.integers(0, len(f), 300)
        out.append(f[i[np.argsort(-p[i])[:40]]].sum() - 40 * 140)
    return np.array(out)

## Урок 11. Итоговый проект: от базы данных до рекомендаций

Финал первой части курса про ML. «Зерно» запустило абонемент и провело пилот: части гостей отправили предложение, части — нет. От вас — проект целиком, как на работе: собрать данные из базы, найти утечку, посчитать экономику, сравнить три модели на кросс-валидации, выбрать порог по деньгам, описать, кому отправлять предложение и сколько это принесёт, и выложить проект на GitHub. Делается в Jupyter; задачи на странице проверяют ключевые промежуточные числа.

Теория — на сайте курса: `ch08/07-final-project.html`. Задачи «собери код из строк» (1) остались на сайте.

### Что лежит в базе · разминка

Подключитесь к `final1.db` и соберите словарь `counts`: ключ — имя таблицы, значение — число строк в ней. Список таблиц возьмите из `sqlite_master`, а не вписывайте руками. Затем сохраните в `n_offer` число участников пилота из группы «предложение», а в `n_july` — число заказов, сделанных 1 июля 2026 года или позже.

*Подсказки (3): `dsc.hint('fp-tables')`*

In [ ]:
%%task ch08/07-final-project/fp-tables
import sqlite3
import pandas as pd

con = sqlite3.connect("final1.db")

### Агрегаты по заказам до среза · база

Одним SQL-запросом к таблице `orders` соберите DataFrame `agg`: одна строка на гостя, у которого были заказы **до 1 июля 2026**, столбцы `guest_id`, `orders_n` (число заказов), `recency_days` (дней от последнего заказа до 1 июля, дробное), `delivery_share` (доля доставки) и `morning_share` (доля заказов в будни до 10 утра). Дату среза передайте параметром `:cutoff`.

*Подсказки (3): `dsc.hint('fp-features')`*

In [ ]:
%%task ch08/07-final-project/fp-features
import sqlite3
import pandas as pd

con = sqlite3.connect("final1.db")
CUTOFF = "2026-07-01"

### Разрезы конверсии · база

Признаки `data` (на 1 июля) и `pilot` уже загружены, группа «предложение» собрана в `off`. Посчитайте конверсию (`bought`) этой группы в трёх разрезах, все доли — 3 знака:

- `by_morning` — по доле утренних будних заказов: группы с подписями `"до 0.2"`, `"0.2–0.5"`, `"выше 0.5"` (границы 0, 0.2, 0.5, 1; ноль входит в первую группу);
- `by_source` — по каналу регистрации `source`;
- `by_city` — по городу.

Затем сохраните в `n_top` число гостей в группе «выше 0.5», а в `lift_top` — во сколько раз их конверсия выше общей конверсии группы «предложение» (2 знака).

*Подсказки (3): `dsc.hint('fp-eda')`*

In [ ]:
%%task ch08/07-final-project/fp-eda
import pandas as pd

off = pilot[pilot["group_name"] == "предложение"].merge(data, on="guest_id", validate="one_to_one")

### Найти утечку · посложнее

Таблица признаков `data` (на 1 июля) и `pilot` уже загружены. Посчитайте по **всей** таблице `orders`, без среза, долю заказов, оплаченных абонементом, у каждого гостя (`sub_share`) и присоедините её к группе «предложение» (гостям без таких заказов — 0). Сохраните ROC-AUC этого признака для `bought` в `auc_leak`, ROC-AUC честного признака `orders_per_week` — в `auc_clean` (оба 3 знака), а в `n_sub_nonbuyers` — сколько **некупивших** из группы «предложение» имеют `sub_share > 0`.

*Подсказки (3): `dsc.hint('fp-leak')`*

In [ ]:
%%task ch08/07-final-project/fp-leak
import sqlite3
import pandas as pd
from sklearn.metrics import roc_auc_score

off = pilot[pilot["group_name"] == "предложение"].merge(data, on="guest_id", validate="one_to_one")
con = sqlite3.connect("final1.db")

### Экономика пилота · база

По таблице `pilot` посчитайте конверсию групп: `conv_offer` и `conv_ctrl` (3 знака), прирост `uplift_pp` в процентных пунктах (1 знак), порог безубыточности `breakeven = C / V` (3 знака). Затем прибыль стратегии «предложение всем» на группе пилота `profit_all` = V × покупатели − C × получатели (целое) и честный итог на одно предложение с поправкой на контроль `honest_per_offer` = (conv_offer − conv_ctrl) × V − C (целое; разность берите из неокруглённых долей). V = 1 600, C = 150.

*Подсказки (3): `dsc.hint('fp-metric')`*

In [ ]:
%%task ch08/07-final-project/fp-metric
import pandas as pd

V, C = 1600, 150
pilot.head()

### Три модели на кросс-валидации · посложнее

Признаки, разбиение и две предобработки (`scaled` — с масштабом, `plain` — без) уже в заготовке. Сравните на обучающей части с одним и тем же `cv` четыре конвейера: `"константа"` (`plain` + `DummyClassifier()`), `"логрегрессия"` (`scaled` + `LogisticRegression(max_iter=1000)`), `"лес"` (`plain` + `RandomForestClassifier(n_estimators=100, max_depth=8, min_samples_leaf=10, random_state=42)`) и `"бустинг"` (`plain` + `HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=200, random_state=42)`). Соберите DataFrame `cv_table` со столбцами `model`, `auc_mean`, `auc_std` (ROC-AUC, 3 знака) в этом порядке строк и сохраните в `best_name` название модели с наибольшим средним. Обучение займёт около минуты.

*Подсказки (3): `dsc.hint('fp-models')`*

In [ ]:
%%task ch08/07-final-project/fp-models
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.compose import make_column_transformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# data (признаки всех гостей на 1 июля) и pilot уже загружены
off = pilot[pilot["group_name"] == "предложение"].merge(data, on="guest_id", validate="one_to_one")
num = ["orders_n", "orders_per_week", "recency_days", "avg_check", "delivery_share", "morning_share",
       "shops_n", "tenure_days", "coffee_share", "app_sessions", "complaints_n"]
cat = ["source", "city"]
X, y = off[num + cat], off["bought"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scaled = make_column_transformer(
    (make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), num),
    (OneHotEncoder(handle_unknown="ignore"), cat))
plain = make_column_transformer(
    (SimpleImputer(strategy="median"), num),
    (OneHotEncoder(handle_unknown="ignore"), cat))
from sklearn.model_selection import cross_val_score
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier

### Порог по прибыли и честная проверка · база

Выбранная модель — логистическая регрессия (`best` в заготовке), функция `profit_curve` — тоже там. Получите прогнозы вне выборки для обучающей части (`cross_val_predict` с тем же `cv`), постройте по ним кривую `curve` и сохраните лучший порог в `best_t`. Затем обучите `best` на всей обучающей части и посчитайте на тесте при пороге `best_t`: число предложений `test_sent`, покупателей среди них `test_buyers` и прибыль `test_profit`; для сравнения — прибыль рассылки всем гостям теста `all_profit`.

*Подсказки (3): `dsc.hint('fp-money')`*

In [ ]:
%%task ch08/07-final-project/fp-money
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.compose import make_column_transformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# data (признаки всех гостей на 1 июля) и pilot уже загружены
off = pilot[pilot["group_name"] == "предложение"].merge(data, on="guest_id", validate="one_to_one")
num = ["orders_n", "orders_per_week", "recency_days", "avg_check", "delivery_share", "morning_share",
       "shops_n", "tenure_days", "coffee_share", "app_sessions", "complaints_n"]
cat = ["source", "city"]
X, y = off[num + cat], off["bought"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scaled = make_column_transformer(
    (make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), num),
    (OneHotEncoder(handle_unknown="ignore"), cat))
plain = make_column_transformer(
    (SimpleImputer(strategy="median"), num),
    (OneHotEncoder(handle_unknown="ignore"), cat))
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_predict

best = make_pipeline(scaled, LogisticRegression(max_iter=1000))

def profit_curve(p, y, value=1600, cost=150):
    rows = []
    for t in np.arange(0.02, 0.41, 0.02).round(2):
        send = p > t
        tp = int((send & (y == 1)).sum())
        rows.append({"threshold": t, "sent": int(send.sum()), "buyers": tp,
                     "profit": value * tp - cost * int(send.sum())})
    return pd.DataFrame(rows)

### Кому отправлять · база

Модель обучена на всех 1 800 гостях группы «предложение» (`final` в заготовке), порог `best_t = 0.08`. Отберите кандидатов осенней рассылки — гостей **не из пилота**, у которых до 1 июля были заказы: DataFrame `rest` со столбцом `p` (вероятность покупки) и `send` (`p > best_t`). Сохраните число кандидатов `n_rest`, число отобранных `n_send`, таблицу `portrait` — средние столбцов `cols` по `send` (индекс False/True, 3 знака) и `app_share` — долю зарегистрированных через приложение среди отобранных (3 знака).

*Подсказки (3): `dsc.hint('fp-portrait')`*

In [ ]:
%%task ch08/07-final-project/fp-portrait
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.compose import make_column_transformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# data (признаки всех гостей на 1 июля) и pilot уже загружены
off = pilot[pilot["group_name"] == "предложение"].merge(data, on="guest_id", validate="one_to_one")
num = ["orders_n", "orders_per_week", "recency_days", "avg_check", "delivery_share", "morning_share",
       "shops_n", "tenure_days", "coffee_share", "app_sessions", "complaints_n"]
cat = ["source", "city"]
X, y = off[num + cat], off["bought"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scaled = make_column_transformer(
    (make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), num),
    (OneHotEncoder(handle_unknown="ignore"), cat))
plain = make_column_transformer(
    (SimpleImputer(strategy="median"), num),
    (OneHotEncoder(handle_unknown="ignore"), cat))
from sklearn.linear_model import LogisticRegression

final = make_pipeline(scaled, LogisticRegression(max_iter=1000)).fit(X, y)   # все 1 800 гостей
best_t = 0.08
cols = ["orders_per_week", "morning_share", "delivery_share", "recency_days", "coffee_share", "app_sessions"]

### Проверка по группам · со звёздочкой

Повторите ориентиры «Проверки по группам» шага 6. В заготовке готовы данные группы «предложение» (`off`, `X`, `y`), `cv`, модель `best` и порог `best_t`. Получите прогнозы вне выборки для всех 1 800 гостей (`cross_val_predict(best, X, y, cv=cv, method="predict_proba")[:, 1]`) и отбор `p > best_t`. Сохраните Series по городам `sel_by_city` (доля отобранных) и `recall_by_city` (доля отобранных среди купивших), 3 знака, и `weak_city` — город с наименьшей полнотой. Затем постройте возрастные группы, как в тексте шага (заглушка 1900 и пропуски — вне групп), и сохраните `recall_by_age` (индекс — группа, 3 знака).

*Подсказки (2): `dsc.hint('fp-groups')` · задача по желанию*

In [ ]:
%%task ch08/07-final-project/fp-groups
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.compose import make_column_transformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# data (признаки всех гостей на 1 июля) и pilot уже загружены
off = pilot[pilot["group_name"] == "предложение"].merge(data, on="guest_id", validate="one_to_one")
num = ["orders_n", "orders_per_week", "recency_days", "avg_check", "delivery_share", "morning_share",
       "shops_n", "tenure_days", "coffee_share", "app_sessions", "complaints_n"]
cat = ["source", "city"]
X, y = off[num + cat], off["bought"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scaled = make_column_transformer(
    (make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), num),
    (OneHotEncoder(handle_unknown="ignore"), cat))
plain = make_column_transformer(
    (SimpleImputer(strategy="median"), num),
    (OneHotEncoder(handle_unknown="ignore"), cat))
from sklearn.linear_model import LogisticRegression

best = make_pipeline(scaled, LogisticRegression(max_iter=1000))   # модель, выбранная в шаге 4
best_t = 0.08

from sklearn.model_selection import cross_val_predict

### Честные деньги рекомендации · со звёздочкой

Наивный расчёт считает всех покупателей заслугой предложения. Поправьте его контрольной группой. Примените `final` к гостям контроля и сохраните в `organic` долю купивших среди тех из них, у кого прогноз выше `best_t` (3 знака). Для кандидатов `rest` посчитайте: `exp_buyers` — ожидаемое число покупок среди отобранных (сумма `p`, 1 знак), `naive_profit` = 1 600 × `exp_buyers` − 150 × `n_send` и `honest_profit` = 1 600 × (сумма p − `organic` × `n_send`) − 150 × `n_send` (оба — целые, по неокруглённым величинам). И для сравнения — честный итог рассылки **всем** кандидатам `honest_all`, с поправкой на общую долю покупок в контроле.

*Подсказки (3): `dsc.hint('fp-honest')` · задача по желанию*

In [ ]:
%%task ch08/07-final-project/fp-honest
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.compose import make_column_transformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# data (признаки всех гостей на 1 июля) и pilot уже загружены
off = pilot[pilot["group_name"] == "предложение"].merge(data, on="guest_id", validate="one_to_one")
num = ["orders_n", "orders_per_week", "recency_days", "avg_check", "delivery_share", "morning_share",
       "shops_n", "tenure_days", "coffee_share", "app_sessions", "complaints_n"]
cat = ["source", "city"]
X, y = off[num + cat], off["bought"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scaled = make_column_transformer(
    (make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), num),
    (OneHotEncoder(handle_unknown="ignore"), cat))
plain = make_column_transformer(
    (SimpleImputer(strategy="median"), num),
    (OneHotEncoder(handle_unknown="ignore"), cat))
from sklearn.linear_model import LogisticRegression

final = make_pipeline(scaled, LogisticRegression(max_iter=1000)).fit(X, y)
best_t = 0.08
rest = data[~data["guest_id"].isin(pilot["guest_id"]) & (data["orders_n"] > 0)].copy()
rest["p"] = final.predict_proba(rest[num + cat])[:, 1]
rest["send"] = rest["p"] > best_t
n_send = int(rest["send"].sum())

## Готово

Запустите `dsc.progress()` — увидите, какие задачи этого сеанса ещё не решены. Перед тем как закрыть блокнот, полезно сделать **Kernel → Restart Kernel and Run All Cells**: если всё проходит сверху вниз, решения не зависят от случайного порядка запуска ячеек.

In [ ]:
dsc.progress()